## Data Generation: IoU Scoring and Interpolation Sweeps

This notebook generates the core CSV artifacts used by downstream plotting and replacement experiments.

### Part 1
Compute IoU scores for 71 programs (70 golden programs + uniform lower diagonal) across BERT, GPT-2, and TinyLlama.

### Part 2
Run linear interpolation sweeps with k = 1..20 top-ranked programs per head.

### Output Files
- ../data/iou_scores_bert.csv
- ../data/iou_scores_gpt2.csv
- ../data/iou_scores_tinyllama.csv
- ../data/interpolation_k_bert.csv
- ../data/interpolation_k_gpt2.csv
- ../data/interpolation_k_tinyllama.csv

In [ ]:
# Install dependencies if needed
# !pip install torch transformers spacy nltk tqdm scikit-learn
# !python -m spacy download en_core_web_sm


In [2]:
import os
import csv
import json
import time
import traceback
import warnings
import inspect
import random
from pathlib import Path
from typing import Callable, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch
import spacy
from tqdm import tqdm
from transformers import (
    AutoTokenizer, AutoModel,
    PreTrainedModel, PreTrainedTokenizerBase
)
from sklearn.linear_model import LinearRegression

warnings.filterwarnings("ignore")

# Reproducibility
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

# Project-relative paths (notebook lives in code/)
DATA_DIR = "../data"
RESULTS_DIR = "../results"
os.makedirs(DATA_DIR, exist_ok=True)
os.makedirs(RESULTS_DIR, exist_ok=True)

nlp = spacy.load("en_core_web_sm")
print(f"[INFO] Imports ready. DATA_DIR={DATA_DIR} | RESULTS_DIR={RESULTS_DIR}")

[INFO] Imports ready. DATA_DIR=../data | RESULTS_DIR=../results


In [3]:
# ── Scoring helpers ──────────────────────────────────────────────────────────

def iou_score(p: np.ndarray, q: np.ndarray) -> float:
    """IoU similarity between two attention distributions (higher = better)."""
    p = np.clip(p, 1e-12, 1.0)
    q = np.clip(q, 1e-12, 1.0)
    intersection = np.sum(np.minimum(p, q))
    union        = np.sum(np.maximum(p, q))
    return float(intersection / union) if union > 0 else 0.0


def score_head_iou(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    layer: int,
    head: int,
    pattern: Callable,
    sentence: str,
) -> float:
    """
    Returns mean row-wise IoU between actual attention and predicted pattern.
    Returns NaN on any error so callers can handle gracefully.
    """
    try:
        tokens = tokenizer(sentence, return_tensors="pt")
        with torch.no_grad():
            att = model(**tokens, output_attentions=True).attentions[layer][0, head].detach().numpy()

        # detect paradigm: 1-arg (Jacob's) vs 2-arg (sentence, tokenizer)
        n_params = len(inspect.signature(pattern).parameters)
        if n_params == 1:
            _, pred_att = pattern(sentence)
        else:
            _, pred_att = pattern(sentence, tokenizer)

        pred_att = np.array(pred_att, dtype=float)
        min_len = min(att.shape[0], pred_att.shape[0])
        att      = att[:min_len, :min_len]
        pred_att = pred_att[:min_len, :min_len]

        row_ious = [iou_score(att[i], pred_att[i]) for i in range(min_len)]
        return float(np.mean(row_ious))

    except Exception:
        return float("nan")


In [6]:
# --- Load golden programs + uniform_lower_diagonal ---
import sys
module_root = os.path.abspath("..")
if module_root not in sys.path:
    sys.path.append(module_root)

from data import golden_programs

all_programs: List[Callable] = [
    obj for _, obj in inspect.getmembers(golden_programs, inspect.isfunction)
]

# Save index mapping so CSV program_idx values are always interpretable
PROGRAM_INDEX_MAP = {i: p.__name__ for i, p in enumerate(all_programs)}
print(f"[INFO] Total programs loaded: {len(all_programs)}")

for idx, name in PROGRAM_INDEX_MAP.items():
    print(f"  {idx}: {name}")
print(f"[INFO] Saving program index map to {DATA_DIR}/program_index_map.json")

with open(f"{DATA_DIR}/program_index_map.json", "w", encoding="utf-8") as _f:
    json.dump(PROGRAM_INDEX_MAP, _f, indent=2)

print("[DONE] Program index map written.")

[INFO] Total programs loaded: 71
  0: adverbial_modulation
  1: appositive_phrase_attention
  2: cls_attention
  3: complement_adjunct_relationship
  4: compound_element_association
  5: compound_modifier_attention
  6: compound_word_attention_pattern
  7: conjunction_based_grouping
  8: conjunction_resolution
  9: contextual_anchoring
  10: coord_and_verb_dependency
  11: coordination_attention
  12: coreference_resolution
  13: dependencies
  14: dominant_subject_attention
  15: emphasize_verbs_and_objects
  16: eos_attention
  17: first_token_dominance
  18: first_token_domination
  19: first_token_emphasis
  20: head_initial_token_emphasis
  21: high_saliency_relationship_detection
  22: initial_contextual_attention
  23: initial_element_reinforcement
  24: initial_phrase_contextualization
  25: initial_phrase_dominance
  26: initial_reference_attention
  27: initial_token_anchoring
  28: initial_token_attachment
  29: initial_token_attention
  30: initial_token_centralization
  31

In [8]:
# --- Load generic sentences and sample 5 (seeded random sample) ---
df_json = pd.read_json(f"{DATA_DIR}/generic_sentences.json")
generic_sentences_all = df_json[0].tolist()

_rng = random.Random(RANDOM_SEED)
_shuffled = generic_sentences_all[:]
_rng.shuffle(_shuffled)
SAMPLE_SENTENCES: List[str] = _shuffled[:5]

# Save mapping so sentence_idx values in CSVs remain recoverable
SENTENCE_INDEX_MAP = {i: s for i, s in enumerate(SAMPLE_SENTENCES)}
print(f"[INFO] Total candidate sentences: {len(generic_sentences_all)}")
print(f"[INFO] Sample size: {len(SAMPLE_SENTENCES)}")
for idx, sentence in SENTENCE_INDEX_MAP.items():
    print(f"  {idx}: {sentence}")
print(f"[INFO] Saving sentence index map to {DATA_DIR}/sentence_index_map.json")

with open(f"{DATA_DIR}/sentence_index_map.json", "w", encoding="utf-8") as _f:
    json.dump(SENTENCE_INDEX_MAP, _f, indent=2)

print("[DONE] Sentence index map written.")

[INFO] Total candidate sentences: 100
[INFO] Sample size: 5
  0: The rhythmic crash of the waves, breaking against the shore, provided a soothing soundtrack to the quiet evening.
  1: He contemplated, for a moment, the vastness of the universe, its endless possibilities, and his tiny place within it.
  2: 'That's an excellent point!' she agreed, nodding enthusiastically, acknowledging the validity of his argument.
  3: If you truly want to succeed, you must work diligently, persevere through challenges, and never give up on your dreams.
  4: He meticulously organized his tools: wrenches, screwdrivers, pliers, and a tape measure, all in their designated spots.
[INFO] Saving sentence index map to ../data/sentence_index_map.json
[DONE] Sentence index map written.


In [9]:
# --- Model configurations ---
MODEL_CONFIGS = {
    "bert": {
        "model_name": "bert-base-uncased",
        "out_csv":    f"{DATA_DIR}/iou_scores_bert.csv",
    },
    "gpt2": {
        "model_name": "openai-community/gpt2",
        "out_csv":    f"{DATA_DIR}/iou_scores_gpt2.csv",
    },
    "tinyllama": {
        "model_name": "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T",
        "out_csv":    f"{DATA_DIR}/iou_scores_tinyllama.csv",
    },
}

print("[INFO] Model configurations loaded.")

[INFO] Model configurations loaded.


In [ ]:
# --- PART 1: IoU scoring (stream rows directly to CSV) ---
# CSV columns: layer, head, program_idx, sentence_idx, iou_score
# Resume-safe: already completed (layer, head, program_idx, sentence_idx) keys are skipped.

PART1_COLUMNS = ["layer", "head", "program_idx", "sentence_idx", "iou_score"]


def load_done_keys(csv_path: str) -> set:
    """Return set of (layer, head, program_idx, sentence_idx) already present in CSV."""
    done = set()
    if not os.path.exists(csv_path):
        return done
    try:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            done.add((int(row["layer"]), int(row["head"]), int(row["program_idx"]), int(row["sentence_idx"])))
    except Exception as e:
        print(f"[WARN] Could not read existing CSV {csv_path}: {e}")
    return done


for model_key, cfg in MODEL_CONFIGS.items():
    model_name = cfg["model_name"]
    out_csv = cfg["out_csv"]

    print(f"\n[INFO] {'=' * 64}")
    print(f"[INFO] Model: {model_key} ({model_name})")
    print(f"[INFO] Output CSV: {out_csv}")
    print(f"[INFO] {'=' * 64}")

    # Load model
    try:
        model_obj = AutoModel.from_pretrained(model_name, output_attentions=True)
        tokenizer_obj = AutoTokenizer.from_pretrained(model_name)
        model_obj.eval()
    except Exception as e:
        print(f"[ERROR] Could not load model {model_name}: {e}")
        traceback.print_exc()
        continue

    num_layers = model_obj.config.num_hidden_layers
    num_heads = model_obj.config.num_attention_heads
    print(f"[INFO] Layers={num_layers} | Heads={num_heads} | Programs={len(all_programs)}")

    done_keys = load_done_keys(out_csv)
    print(f"[INFO] Resume keys loaded: {len(done_keys)}")

    write_header = not os.path.exists(out_csv) or os.path.getsize(out_csv) == 0
    errors = 0

    with open(out_csv, "a", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        if write_header:
            writer.writerow(PART1_COLUMNS)
            fh.flush()

        total = num_layers * num_heads * len(all_programs) * len(SAMPLE_SENTENCES)
        done_count = len(done_keys)
        t0 = time.time()

        for p_idx, prog in enumerate(all_programs):
            for layer in range(num_layers):
                for head in range(num_heads):
                    for s_idx, sentence in enumerate(SAMPLE_SENTENCES):
                        key = (layer, head, p_idx, s_idx)
                        if key in done_keys:
                            done_count += 1
                            continue

                        score = score_head_iou(model_obj, tokenizer_obj, layer, head, prog, sentence)
                        score_out = round(score, 3) if not np.isnan(score) else score
                        writer.writerow([layer, head, p_idx, s_idx, score_out])
                        fh.flush()
                        done_count += 1

                        if np.isnan(score):
                            errors += 1

                        if done_count % 720 == 0:
                            elapsed = time.time() - t0
                            pct = done_count / total * 100
                            eta = elapsed / done_count * (total - done_count) if done_count else 0
                            print(
                                f"[INFO] [{model_key}] {done_count}/{total} ({pct:.1f}%) | "
                                f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | NaN={errors}"
                            )

    del model_obj, tokenizer_obj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"[DONE] {model_key} complete. NaN scores: {errors}")
    df_check = pd.read_csv(out_csv)
    print(f"[INFO] Rows currently in CSV: {len(df_check)}")

print("\n[DONE] Part 1 complete.")


  Model: bert  (bert-base-uncased)
  Output: data/iou_scores_bert.csv
  Layers: 12  Heads: 12  Programs: 71
  Already done rows: 0
  [bert] 720/51120 (1.4%) | elapsed 0.6m | ETA 42.1m | errors 0
  [bert] 1440/51120 (2.8%) | elapsed 1.1m | ETA 38.7m | errors 0
  [bert] 2160/51120 (4.2%) | elapsed 2.0m | ETA 44.8m | errors 0
  [bert] 2880/51120 (5.6%) | elapsed 2.9m | ETA 49.0m | errors 0
  [bert] 3600/51120 (7.0%) | elapsed 4.2m | ETA 55.5m | errors 0
  [bert] 4320/51120 (8.5%) | elapsed 11.8m | ETA 127.8m | errors 0
  [bert] 5040/51120 (9.9%) | elapsed 12.3m | ETA 112.6m | errors 0
  [bert] 5760/51120 (11.3%) | elapsed 12.9m | ETA 101.8m | errors 0
  [bert] 6480/51120 (12.7%) | elapsed 13.5m | ETA 93.0m | errors 0
  [bert] 7200/51120 (14.1%) | elapsed 14.0m | ETA 85.5m | errors 0
  [bert] 7920/51120 (15.5%) | elapsed 14.5m | ETA 79.2m | errors 0
  [bert] 8640/51120 (16.9%) | elapsed 15.0m | ETA 74.0m | errors 0
  [bert] 9360/51120 (18.3%) | elapsed 15.6m | ETA 69.8m | errors 0
  [bert

In [10]:
# --- Part 1 verification ---
for model_key, cfg in MODEL_CONFIGS.items():
    csv_path = cfg["out_csv"]
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
        print(
            f"[INFO] {model_key}: rows={len(df)} | NaN iou={df['iou_score'].isna().sum()} | "
            f"programs={df['program_idx'].nunique()} | mean_iou={df['iou_score'].mean():.4f}"
        )
    else:
        print(f"[WARN] {model_key}: CSV not found at {csv_path}")

[INFO] bert: rows=51120 | NaN iou=0 | programs=71 | mean_iou=0.1090
[INFO] gpt2: rows=51120 | NaN iou=288 | programs=71 | mean_iou=0.2613
[INFO] tinyllama: rows=237401 | NaN iou=0 | programs=68 | mean_iou=0.2909


## Part 2: Linear Interpolation Sweep (k = 1..20)

For each model and each attention head:
1. Rank programs by Part 1 mean IoU
2. Fit linear combinations using the top-k programs
3. Record IoU for each k

Output columns:
- model_key
- layer
- head
- k
- iou_score
- program_idxs_used

In [ ]:
# --- Part 2 helpers ---

def linear_interp_iou(
    model: PreTrainedModel,
    tokenizer: PreTrainedTokenizerBase,
    layer: int,
    head: int,
    top_k_programs: List[Callable],
    sentences: List[str],
) -> float:
    """
    Fit a linear combination of top_k_programs to minimize squared error vs
    actual attention, then measure mean IoU of the fitted combination.
    Returns NaN on failure.
    """
    all_ious = []

    for sentence in sentences:
        try:
            tokens = tokenizer(sentence, return_tensors="pt")
            with torch.no_grad():
                att = model(**tokens, output_attentions=True).attentions[layer][0, head].detach().numpy()

            # Build feature matrix: each program contributes one flattened attention matrix
            X_rows = []
            for prog in top_k_programs:
                try:
                    n_params = len(inspect.signature(prog).parameters)
                    _, pred = prog(sentence) if n_params == 1 else prog(sentence, tokenizer)
                    pred = np.array(pred, dtype=float)
                    min_len = min(att.shape[0], pred.shape[0])
                    pred = pred[:min_len, :min_len]
                    X_rows.append(pred.flatten())
                except Exception:
                    return float("nan")

            min_len = min(att.shape[0], len(X_rows[0]) ** 0.5)
            seq_len = att.shape[0]

            flat_len = seq_len * seq_len
            X_mat = np.array([r[:flat_len] for r in X_rows]).T
            y_vec = att.flatten()[:flat_len]

            X_mat = np.nan_to_num(X_mat)
            y_vec = np.nan_to_num(y_vec)

            reg = LinearRegression(fit_intercept=True).fit(X_mat, y_vec)
            fitted = reg.intercept_ + X_mat @ reg.coef_
            fitted = fitted.reshape(seq_len, seq_len)

            row_sums = fitted.sum(axis=1, keepdims=True)
            row_sums = np.where(row_sums == 0, 1, row_sums)
            fitted = fitted / row_sums
            fitted = np.clip(fitted, 1e-12, None)

            row_ious = [iou_score(att[i], fitted[i]) for i in range(seq_len)]
            all_ious.append(float(np.mean(row_ious)))

        except Exception:
            all_ious.append(float("nan"))

    valid = [v for v in all_ious if not np.isnan(v)]
    return float(np.mean(valid)) if valid else float("nan")


K_VALUES = list(range(1, 21))
PART2_COLUMNS = ["model_key", "layer", "head", "k", "iou_score", "program_idxs_used"]

print(f"[INFO] Part 2 K values: {K_VALUES}")
print(f"[INFO] Part 2 columns: {PART2_COLUMNS}")

In [ ]:
# --- PART 2: Linear interpolation sweep ---
# Strategy:
# 1) Use Part 1 CSV to rank programs per head by mean IoU
# 2) For each head and k in 1..20, fit linear combinations of top-k programs
# 3) Stream each row to CSV
# Resume-safe: skips (layer, head, k) that are already present.

INTERP_CONFIGS = {
    "bert":      f"{DATA_DIR}/interpolation_k_bert.csv",
    "gpt2":      f"{DATA_DIR}/interpolation_k_gpt2.csv",
    "tinyllama": f"{DATA_DIR}/interpolation_k_tinyllama.csv",
}


def load_done_interp(csv_path: str) -> set:
    done = set()
    if not os.path.exists(csv_path):
        return done
    try:
        df = pd.read_csv(csv_path)
        for _, row in df.iterrows():
            done.add((int(row["layer"]), int(row["head"]), int(row["k"])))
    except Exception as e:
        print(f"[WARN] Could not read {csv_path}: {e}")
    return done


for model_key in MODEL_CONFIGS:
    iou_csv = MODEL_CONFIGS[model_key]["out_csv"]
    out_csv = INTERP_CONFIGS[model_key]
    model_name = MODEL_CONFIGS[model_key]["model_name"]

    print(f"\n[INFO] {'=' * 64}")
    print(f"[INFO] Model: {model_key} ({model_name})")
    print(f"[INFO] Input Part 1 CSV: {iou_csv}")
    print(f"[INFO] Output Part 2 CSV: {out_csv}")
    print(f"[INFO] {'=' * 64}")

    if not os.path.exists(iou_csv):
        print(f"[WARN] Missing Part 1 CSV: {iou_csv}. Skipping.")
        continue

    iou_df = pd.read_csv(iou_csv)
    if iou_df.empty:
        print(f"[WARN] Part 1 CSV is empty for {model_key}. Skipping.")
        continue

    try:
        model_obj = AutoModel.from_pretrained(model_name, output_attentions=True)
        tokenizer_obj = AutoTokenizer.from_pretrained(model_name)
        model_obj.eval()
    except Exception as e:
        print(f"[ERROR] Could not load model {model_name}: {e}")
        traceback.print_exc()
        continue

    num_layers = model_obj.config.num_hidden_layers
    num_heads = model_obj.config.num_attention_heads

    # Mean IoU per (layer, head, program_idx), then rank programs per head
    head_rankings = {}
    grouped = (
        iou_df.groupby(["layer", "head", "program_idx"])["iou_score"]
        .mean()
        .reset_index()
    )
    for (layer, head), grp in grouped.groupby(["layer", "head"]):
        sorted_grp = grp.sort_values("iou_score", ascending=False)
        ordered_fns = []
        for p_idx in sorted_grp["program_idx"]:
            fn = all_programs[int(p_idx)] if int(p_idx) < len(all_programs) else None
            if fn is not None:
                ordered_fns.append(fn)
        head_rankings[(int(layer), int(head))] = ordered_fns

    total = num_layers * num_heads * len(K_VALUES)
    done_keys = load_done_interp(out_csv)
    print(f"[INFO] Resume keys loaded: {len(done_keys)} | Total target rows: {total}")

    write_header = not os.path.exists(out_csv) or os.path.getsize(out_csv) == 0
    errors = 0
    done_count = len(done_keys)
    t0 = time.time()

    with open(out_csv, "a", newline="", encoding="utf-8") as fh:
        writer = csv.writer(fh)
        if write_header:
            writer.writerow(PART2_COLUMNS)
            fh.flush()

        for layer in range(num_layers):
            for head in range(num_heads):
                ranked_fns = head_rankings.get((layer, head), all_programs)

                for k in K_VALUES:
                    key = (layer, head, k)
                    if key in done_keys:
                        done_count += 1
                        continue

                    top_k_fns = ranked_fns[:k]
                    if not top_k_fns:
                        top_k_fns = all_programs[:k]

                    iou = linear_interp_iou(
                        model_obj,
                        tokenizer_obj,
                        layer,
                        head,
                        top_k_fns,
                        SAMPLE_SENTENCES,
                    )

                    prog_idx_used = "|".join(str(all_programs.index(f)) for f in top_k_fns)
                    iou_out = round(iou, 6) if not np.isnan(iou) else iou
                    writer.writerow([model_key, layer, head, k, iou_out, prog_idx_used])
                    fh.flush()
                    done_count += 1

                    if np.isnan(iou):
                        errors += 1

                    if done_count % 200 == 0:
                        elapsed = time.time() - t0
                        pct = done_count / total * 100
                        eta = elapsed / done_count * (total - done_count) if done_count else 0
                        print(
                            f"[INFO] [{model_key}] {done_count}/{total} ({pct:.1f}%) | "
                            f"elapsed {elapsed/60:.1f}m | ETA {eta/60:.1f}m | NaN={errors}"
                        )

    del model_obj, tokenizer_obj
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    print(f"[DONE] {model_key} interpolation complete. NaN scores: {errors}")
    df_check = pd.read_csv(out_csv)
    print(f"[INFO] Rows currently in CSV: {len(df_check)}")

print("\n[DONE] Part 2 complete.")

In [13]:
# --- Final verification summary ---
print("[INFO] === Part 1: IoU Scores ===")
for model_key, cfg in MODEL_CONFIGS.items():
    p = cfg["out_csv"]
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(
            f"[INFO] {model_key}: rows={len(df)} | programs={df['program_idx'].nunique()} | "
            f"mean_iou={df['iou_score'].mean():.4f} | NaN={df['iou_score'].isna().sum()}"
        )
    else:
        print(f"[WARN] {model_key}: file not found at {p}")

print("\n[INFO] === Part 2: Interpolation Sweep ===")
for model_key, p in INTERP_CONFIGS.items():
    if os.path.exists(p):
        df = pd.read_csv(p)
        print(
            f"[INFO] {model_key}: rows={len(df)} | k_range={df['k'].min()}..{df['k'].max()} | "
            f"mean_iou={df['iou_score'].mean():.4f} | NaN={df['iou_score'].isna().sum()}"
        )
    else:
        print(f"[WARN] {model_key}: file not found at {p}")

print("\n[DONE] Verification summary complete.")

[INFO] === Part 1: IoU Scores ===
[INFO] bert: rows=51120 | programs=71 | mean_iou=0.1090 | NaN=0
[INFO] gpt2: rows=51120 | programs=71 | mean_iou=0.2613 | NaN=288
[INFO] tinyllama: rows=237401 | programs=68 | mean_iou=0.2909 | NaN=0

[INFO] === Part 2: Interpolation Sweep ===
[INFO] bert: rows=2880 | k_range=1..20 | mean_iou=0.4760 | NaN=0
[INFO] gpt2: rows=2880 | k_range=1..20 | mean_iou=0.5893 | NaN=8
[INFO] tinyllama: rows=13088 | k_range=1..20 | mean_iou=0.6961 | NaN=0

[DONE] Verification summary complete.
